# 🫁 Respiratory Sound Analysis

> Analysis of respiratory sounds using advanced signal processing techniques

---

## 📦 Installation & Setup

In [ ]:
!pip install -q kaggle numpy scipy matplotlib pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## ⚙️ Configure Kaggle & Download Dataset

In [ ]:
import os
import shutil
from google.colab import files

KAGGLE_JSON_DRIVE = '/content/drive/MyDrive/kaggle.json'
KAGGLE_JSON_LOCAL = os.path.expanduser('~/.kaggle/kaggle.json')

if os.path.exists(KAGGLE_JSON_DRIVE):
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    shutil.copy(KAGGLE_JSON_DRIVE, KAGGLE_JSON_LOCAL)
    os.chmod(KAGGLE_JSON_LOCAL, 0o600)
else:
    uploaded = files.upload()
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(KAGGLE_JSON_LOCAL, 'wb') as f:
        f.write(list(uploaded.values())[0])
    os.chmod(KAGGLE_JSON_LOCAL, 0o600)
    shutil.copy(KAGGLE_JSON_LOCAL, KAGGLE_JSON_DRIVE)

In [ ]:
DATASET_PATH = '/content/respiratory_sound_dataset'

print("📥 Downloading dataset from Kaggle...")
!kaggle datasets download -d vbookshelf/respiratory-sound-database
!unzip -q respiratory-sound-database.zip -d {DATASET_PATH}
!rm respiratory-sound-database.zip

print(f"✅ Dataset downloaded")

## 📥 Clone Analysis Repository

In [ ]:
import sys

REPO_PATH = '/content/course_paper'

if os.path.exists(REPO_PATH):
    print(f"✅ Repository already cloned")
else:
    print("📥 Cloning repository...")
    !git clone --depth 1 --filter=blob:none --sparse https://github.com/incRED1bl/course_paper.git {REPO_PATH}
    !git -C {REPO_PATH} sparse-checkout set app
    print("✅ Repository cloned")

sys.path.insert(0, REPO_PATH)

In [ ]:
import os
import numpy as np
from scipy.io import wavfile
from pathlib import Path

def load_respiratory_sounds(dataset_path, max_files=20):
    """Load audio files from the dataset efficiently."""
    dataset_path = Path(dataset_path)
    
    if not dataset_path.exists():
        raise FileNotFoundError(f"Path not found: {dataset_path}")
    
    audio_dir = None
    for pattern in ['**/audio_and_txt_files', '**/Respiratory_Sound_Database/**/audio_and_txt_files']:
        matches = list(dataset_path.glob(pattern))
        if matches:
            audio_dir = matches[0]
            break
    
    if not audio_dir or not audio_dir.exists():
        raise FileNotFoundError(f"audio_and_txt_files folder not found in {dataset_path}")
    
    wav_files = sorted(audio_dir.glob("*.wav"))
    load_count = min(max_files, len(wav_files))
    
    signals = {}
    load_errors = 0
    
    for wav_file in wav_files[:load_count]:
        try:
            sample_rate, signal_data = wavfile.read(str(wav_file))
            if signal_data.dtype != np.float64:
                signal_data = signal_data.astype(np.float64)
            if signal_data.ndim > 1:
                signal_data = signal_data[:, 0]
            
            signals[wav_file.name] = {
                'signal': signal_data,
                'sample_rate': sample_rate
            }
        except Exception as e:
            load_errors += 1
            if load_errors <= 3:
                print(f"⚠️ Failed: {wav_file.name}")
    
    if load_errors > 3:
        print(f"⚠️ ...and {load_errors - 3} more errors")
    
    return signals

## 🔊 Extract Features and Create DataFrame

In [ ]:
signals = load_respiratory_sounds(DATASET_PATH, max_files=20)
sample_rate = list(signals.values())[0]['sample_rate']

print(f"✅ Loaded {len(signals)} files @ {sample_rate} Hz")

In [ ]:
import pandas as pd
from pathlib import Path
from app.data_preprocessing import extract_features_batch

diagnosis_path = Path(DATASET_PATH)
diagnosis_files = list(diagnosis_path.rglob('patient_diagnosis.csv'))

if not diagnosis_files:
    raise FileNotFoundError(f"patient_diagnosis.csv not found in {diagnosis_path}")

diagnosis_file = diagnosis_files[0]
print(f"📋 Found: {diagnosis_file}")

diagnosis_df = pd.read_csv(diagnosis_file)
patient_col, diagnosis_col = diagnosis_df.columns[0], diagnosis_df.columns[1]
diagnosis_map = dict(zip(diagnosis_df[patient_col], diagnosis_df[diagnosis_col]))

rows = extract_features_batch(signals, embedding_dim=3, time_delay=1)

for row in rows:
    row['diagnosis'] = diagnosis_map.get(row['patient_id'], 'Unknown')

results_df = pd.DataFrame(rows)

print(f"\n✅ Extracted features: {results_df.shape}")
print(f"\n📋 Diagnoses:\n{results_df['diagnosis'].value_counts()}")
display(results_df.head())

In [ ]:
feature_cols = [
    'low_freq_energy', 'mid_freq_energy', 'high_freq_energy',
    'whistle_strength', 'spectral_centroid', 'peak_frequency',
    'entropy', 'complexity'
]

avg_by_diagnosis = results_df.groupby('diagnosis')[feature_cols].mean().reset_index()


display(avg_by_diagnosis)

## 📊 Visualizations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
DIAGNOSIS_COLORS = ['#FF4757', '#1DD1A1', '#5F9EFF', '#FFA502', '#00D2C3', '#FFE66D', '#C56CF0', '#54A0FF']
PLOT_STYLE = {
    'grid_alpha': 0.15,
    'bar_alpha': 0.85,
    'line_width': 2.0,
    'title_fontsize': 15,
    'label_fontsize': 12,
    'value_fontsize': 9
}

plt.style.use('dark_background')

plt.rcParams['figure.facecolor'] = '#0B1929'
plt.rcParams['axes.facecolor'] = '#1A2332'
plt.rcParams['savefig.facecolor'] = '#0B1929'


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, feature in enumerate(feature_cols):
    ax = axes[idx]
    
    diagnoses = avg_by_diagnosis['diagnosis']
    values = avg_by_diagnosis[feature]
    
    colors = [DIAGNOSIS_COLORS[i % len(DIAGNOSIS_COLORS)] for i in range(len(diagnoses))]
    bars = ax.bar(range(len(diagnoses)), values, color=colors, alpha=PLOT_STYLE['bar_alpha'], width=0.7)
    
    for i, (bar, val) in enumerate(zip(bars, values)):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                f'{val:.3f}', ha='center', va='bottom', fontsize=PLOT_STYLE['value_fontsize'],
                fontweight='600')
    
    ax.set_title(feature.replace('_', ' ').title(), fontsize=11, fontweight='bold', pad=10)
    ax.set_xticks(range(len(diagnoses)))
    ax.set_xticklabels(diagnoses, rotation=45, ha='right', fontsize=PLOT_STYLE['value_fontsize'])
    ax.grid(True, alpha=PLOT_STYLE['grid_alpha'], axis='y', zorder=0)
    ax.set_ylim(0, values.max() * 1.20)
    ax.set_axisbelow(True)

plt.suptitle('Feature Comparison Across Diagnoses', fontsize=PLOT_STYLE['title_fontsize']+2, 
             fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()


In [ ]:
min_length = min(len(data['signal']) for data in signals.values())

signals_by_diagnosis = {}
for filename, data in signals.items():
    patient_id = int(filename.split('_')[0])
    diagnosis = diagnosis_map.get(patient_id, 'Unknown')
    
    signal = data['signal'][:min_length]
    
    if diagnosis not in signals_by_diagnosis:
        signals_by_diagnosis[diagnosis] = []
    signals_by_diagnosis[diagnosis].append(signal)

diagnoses = avg_by_diagnosis['diagnosis'].tolist()

for idx, diagnosis in enumerate(diagnoses):
    if diagnosis in signals_by_diagnosis and len(signals_by_diagnosis[diagnosis]) > 0:
        signals_array = np.array(signals_by_diagnosis[diagnosis])
        
        mean_signal = np.mean(signals_array, axis=0)
        std_signal = np.std(signals_array, axis=0)
        time_points = np.arange(len(mean_signal))
        
        file_count = len(signals_by_diagnosis[diagnosis])
        
        color = DIAGNOSIS_COLORS[idx % len(DIAGNOSIS_COLORS)]
        
        fig, ax = plt.subplots(figsize=(12, 4.5))
        
        ax.fill_between(time_points, 
                        mean_signal - std_signal, 
                        mean_signal + std_signal, 
                        color=color, alpha=0.3, label='±1 SD', zorder=2)
        
        ax.plot(time_points, mean_signal, color=color, linewidth=PLOT_STYLE['line_width']+0.5, 
                label='Mean', alpha=0.95, zorder=3)
        
        marker_interval = len(time_points) // 20
        if marker_interval > 0:
            ax.plot(time_points[::marker_interval], mean_signal[::marker_interval], 
                   'o', color=color, markersize=4, zorder=4)
        
        ax.set_title(f'{diagnosis} - Average Time Domain Signal (n={file_count}, samples={len(mean_signal)})', 
                    fontsize=PLOT_STYLE['title_fontsize'], fontweight='bold', pad=15)
        ax.set_xlabel('Time (samples)', fontsize=PLOT_STYLE['label_fontsize'], fontweight='500')
        ax.set_ylabel('Amplitude', fontsize=PLOT_STYLE['label_fontsize'], fontweight='500')
        ax.grid(True, alpha=PLOT_STYLE['grid_alpha'], zorder=0)
        ax.set_axisbelow(True)
        
        legend = ax.legend(loc='upper right', fontsize=10, framealpha=0.95, fancybox=True, shadow=False)
        legend.get_frame().set_linewidth(1.0)
        
        ax.legend(loc='upper right', fontsize=10, framealpha=0.9)
        
        plt.tight_layout()
        plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))

for idx, row in avg_by_diagnosis.iterrows():
    diagnosis = row['diagnosis']
    entropy = row['entropy']
    complexity = row['complexity']
    color = DIAGNOSIS_COLORS[idx % len(DIAGNOSIS_COLORS)]
    
    ax.scatter(entropy, complexity, s=300, color=color, alpha=PLOT_STYLE['bar_alpha'], 
              label=diagnosis, zorder=3)
    
    ax.annotate(diagnosis, (entropy, complexity), 
               xytext=(10, 10), textcoords='offset points',
               fontsize=10, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.5', facecolor=color, alpha=0.3),
               arrowprops=dict(arrowstyle='-', lw=0.8, alpha=0.6))

ax.set_xlabel('Entropy', fontsize=13, fontweight='bold', labelpad=10)
ax.set_ylabel('Complexity', fontsize=13, fontweight='bold', labelpad=10)
ax.set_title('Entropy vs Complexity by Diagnosis', fontsize=16, fontweight='bold', pad=20)
ax.grid(True, alpha=PLOT_STYLE['grid_alpha'], linestyle='--', zorder=0)
ax.set_axisbelow(True)

legend = ax.legend(loc='best', fontsize=10, framealpha=0.95, fancybox=True, shadow=False)
legend.get_frame().set_linewidth(1.2)

plt.tight_layout()
plt.show()
